# Hosting Strands Agents with Amazon Bedrock models in Amazon Bedrock AgentCore Runtime
# 在 Amazon Bedrock AgentCore Runtime 中托管使用 Amazon Bedrock 模型的 Strands 智能体

## Overview
## 概述

In this tutorial we will learn how to host your existing agent, using Amazon Bedrock AgentCore Runtime. We will provide examples using Amazon Bedrock models and non-Bedrock models such as Azure OpenAI and Gemini.

在本教程中，我们将学习如何使用 Amazon Bedrock AgentCore Runtime 托管您现有的智能体。我们将提供使用 Amazon Bedrock 模型以及非 Bedrock 模型（如 Azure OpenAI 和 Gemini）的示例。


### Tutorial Details
### 教程详情


| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                        |
| Tutorial components | Hosting agent on AgentCore Runtime. Using Strands Agent and Amazon Bedrock Model |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK and boto3                                     |

| 信息项         | 详情                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 教程类型       | 对话式                                                                   |
| 智能体类型          | 单一智能体                                                                           |
| 智能体框架   | Strands Agents                                                                   |
| LLM 模型           | Anthropic Claude Haiku 4.5                                                        |
| 教程组件 | 在 AgentCore Runtime 上托管智能体，使用 Strands Agent 和 Amazon Bedrock 模型 |
| 教程适用领域   | 跨领域                                                                   |
| 示例复杂度  | 简单                                                                             |
| 使用的 SDK            | Amazon BedrockAgentCore Python SDK 和 boto3                                     |

### Tutorial Architecture
### 教程架构

In this tutorial we will describe how to deploy an existing agent to AgentCore runtime. 

在本教程中，我们将介绍如何将现有智能体部署到 AgentCore Runtime。

For demonstration purposes, we will  use a Strands Agent using Amazon Bedrock models

出于演示目的，我们将使用一个基于 Amazon Bedrock 模型的 Strands Agent。

In our example we will use a very simple agent with two tools: `get_weather` and `get_time`. 

在我们的示例中，我们将使用一个非常简单的智能体，它包含两个工具：`get_weather` 和 `get_time`。

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### Tutorial Key Features
### 教程核心特性

* Hosting Agents on Amazon Bedrock AgentCore Runtime
* Using Amazon Bedrock models
* Using Strands Agents

* 在 Amazon Bedrock AgentCore Runtime 上托管智能体
* 使用 Amazon Bedrock 模型
* 使用 Strands Agents 框架

## Prerequisites
## 前提条件

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Strands Agents

要执行本教程，您需要：
* Python 3.10+
* AWS 凭证
* Amazon Bedrock AgentCore SDK
* Strands Agents

In [1]:
%pip install --force-reinstall -U -r requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Creating your agents and experimenting locally
## 创建智能体并在本地进行实验

Before we deploy our agents to AgentCore Runtime, let's develop and run them locally for experimentation purposes.

在将智能体部署到 AgentCore Runtime 之前，让我们先在本地开发和运行它们进行实验。

For production agentic applications we will need to decouple the agent creation process from the agent invocation one. With AgentCore Runtime, we will decorate the invocation part of our agent with the `@app.entrypoint` decorator and have it as the entry point for our runtime. Let's first look how each agent is developed during the experimentation phase.

对于生产级的智能体应用，我们需要将智能体的创建过程与调用过程解耦。使用 AgentCore Runtime 时，我们将使用 `@app.entrypoint` 装饰器来装饰智能体的调用部分，并将其作为运行时的入口点。让我们先看看在实验阶段每个智能体是如何开发的。

The architecture here will look as following:

这里的架构如下所示：

<div style="text-align:left">
    <img src="images/architecture_local.png" width="50%"/>
</div>

In [10]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from strands.models import BedrockModel

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"


model_id = "qwen.qwen3-32b-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = strands_agent_bedrock(json.loads(args.payload))

Overwriting strands_claude.py


#### Invoking local agent
#### 调用本地智能体

In [11]:
!python strands_claude.py "{\"prompt\": \"What is the weather now?\"}"
# !python strands_claude.py "{\"prompt\": \"5+2*2?\"}"


The current weather is sunny! ☀️


## Preparing your agent for deployment on AgentCore Runtime
## 准备将智能体部署到 AgentCore Runtime

Let's now deploy our agents to AgentCore Runtime. To do so we need to:
* Import the Runtime App with `from bedrock_agentcore.runtime import BedrockAgentCoreApp`
* Initialize the App in our code with `app = BedrockAgentCoreApp()`
* Decorate the invocation function with the `@app.entrypoint` decorator
* Let AgentCoreRuntime control the running of the agent with `app.run()`

现在让我们将智能体部署到 AgentCore Runtime。为此我们需要：
* 使用 `from bedrock_agentcore.runtime import BedrockAgentCoreApp` 导入 Runtime App
* 在代码中使用 `app = BedrockAgentCoreApp()` 初始化 App
* 使用 `@app.entrypoint` 装饰器装饰调用函数
* 使用 `app.run()` 让 AgentCore Runtime 控制智能体的运行

### Strands Agents with Amazon Bedrock model
### 使用 Amazon Bedrock 模型的 Strands Agents

Let's start with our Strands Agent using Amazon Bedrock model. All the others will work exactly the same.

让我们从使用 Amazon Bedrock 模型的 Strands Agent 开始。其他所有的智能体都将以完全相同的方式工作。

In [12]:
%%writefile strands_claude.py
from strands import Agent, tool
from strands_tools import calculator # Import the calculator tool
import argparse
import json
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.models import BedrockModel

app = BedrockAgentCoreApp()

# Create a custom tool 
@tool
def weather():
    """ Get weather """ # Dummy implementation
    return "sunny"


model_id = "qwen.qwen3-32b-v1:0"
model = BedrockModel(
    model_id=model_id,
)
agent = Agent(
    model=model,
    tools=[calculator, weather],
    system_prompt="You're a helpful assistant. You can do simple math calculation, and tell the weather."
)

@app.entrypoint
def strands_agent_bedrock(payload):
    """
    Invoke the agent with a payload
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == "__main__":
    app.run()

Overwriting strands_claude.py


## What happens behind the scenes?
## 幕后发生了什么？

When you use `BedrockAgentCoreApp`, it automatically:

当您使用 `BedrockAgentCoreApp` 时，它会自动：

* Creates an HTTP server that listens on the port 8080
* Implements the required `/invocations` endpoint for processing the agent's requirements
* Implements the `/ping` endpoint for health checks (very important for asynchronous agents)
* Handles proper content types and response formats
* Manages error handling according to the AWS standards

* 创建一个监听 8080 端口的 HTTP 服务器
* 实现处理智能体请求所需的 `/invocations` 端点
* 实现用于健康检查的 `/ping` 端点（对于异步智能体非常重要）
* 处理正确的内容类型和响应格式
* 按照 AWS 标准进行错误处理

## Deploying the agent to AgentCore Runtime
## 将智能体部署到 AgentCore Runtime

The `CreateAgentRuntime` operation supports comprehensive configuration options, letting you specify container images, environment variables and encryption settings. You can also configure protocol settings (HTTP, MCP) and authorization mechanisms to control how your clients communicate with the agent. 

`CreateAgentRuntime` 操作支持全面的配置选项，允许您指定容器镜像、环境变量和加密设置。您还可以配置协议设置（HTTP、MCP）和授权机制，以控制客户端如何与智能体通信。

**Note:** Operations best practice is to package code as container and push to ECR using CI/CD pipelines and IaC

**注意：** 运维最佳实践是将代码打包为容器，并使用 CI/CD 流水线和基础设施即代码（IaC）推送到 ECR

In this tutorial can will the Amazon Bedrock AgentCore Python SDK to easily package your artifacts and deploy them to AgentCore runtime.

在本教程中，我们将使用 Amazon Bedrock AgentCore Python SDK 轻松打包您的工件并将其部署到 AgentCore Runtime。

### Configure AgentCore Runtime deployment
### 配置 AgentCore Runtime 部署

First we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

首先，我们将使用启动工具包来配置 AgentCore Runtime 部署，包括入口点、我们刚创建的执行角色和依赖文件。我们还将配置启动工具包在启动时自动创建 Amazon ECR 仓库。

During the configure step, your docker file will be generated based on your application code

在配置步骤中，将根据您的应用程序代码生成 Dockerfile。

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [13]:
from bedrock_agentcore_starter_toolkit import Runtime
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "strands_claude_getting_started"
response = agentcore_runtime.configure(
    entrypoint="strands_claude.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
)
response

Entrypoint parsed: file=D:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\strands_claude.py, bedrock_agentcore_name=strands_claude
Configuring BedrockAgentCore agent: strands_claude_getting_started


⚠️  [WARNING] Platform mismatch: Current system is 'linux/amd64' but Bedrock AgentCore requires 'linux/arm64'.
For deployment options and workarounds, see: 
https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/getting-started-custom.html

Generated .dockerignore
Generated Dockerfile: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\Dockerfile
Generated .dockerignore: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\.dockerignore
Setting 'strands_claude_getting_started' as default agent
Bedrock AgentCore configured: d:\Documents\amazon-bedrock-agentcore-samples\01-tutorials\01-AgentCore-runtime\01-hosting-agent\01-strands-with-bedrock-model\.bedrock_agentcore.yaml


ConfigureResult(config_path=WindowsPath('d:/Documents/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/.bedrock_agentcore.yaml'), dockerfile_path=WindowsPath('d:/Documents/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/Dockerfile'), dockerignore_path=WindowsPath('d:/Documents/amazon-bedrock-agentcore-samples/01-tutorials/01-AgentCore-runtime/01-hosting-agent/01-strands-with-bedrock-model/.dockerignore'), runtime='Docker', region='us-east-1', account_id='710560201993', execution_role=None, ecr_repository=None, auto_create_ecr=True)

### Launching agent to AgentCore Runtime
### 将智能体启动到 AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

现在我们已经有了 Dockerfile，让我们将智能体启动到 AgentCore Runtime。这将创建 Amazon ECR 仓库和 AgentCore Runtime。

<div style="text-align:left">
    <img src="images/launch.png" width="75%"/>
</div>

In [14]:
launch_result = agentcore_runtime.launch()

🚀 CodeBuild mode: building in cloud (RECOMMENDED - DEFAULT)
   • Build ARM64 containers in the cloud with CodeBuild
   • No local Docker required
💡 Available deployment modes:
   • runtime.launch()                           → CodeBuild (current)
   • runtime.launch(local=True)                 → Local development
   • runtime.launch(local_build=True)           → Local build + cloud deploy (NEW)
Starting CodeBuild ARM64 deployment for agent 'strands_claude_getting_started' to account 710560201993 (us-east-1)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: strands_claude_getting_started


Repository doesn't exist, creating new ECR repository: bedrock-agentcore-strands_claude_getting_started


✅ ECR repository available: 710560201993.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_claude_getting_started
Getting or creating execution role for agent: strands_claude_getting_started
Using AWS region: us-east-1, account ID: 710560201993
Role name: AmazonBedrockAgentCoreSDKRuntime-us-east-1-8556fc4504
Role doesn't exist, creating new execution role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-8556fc4504
Starting execution role creation process for agent: strands_claude_getting_started
✓ Role creating: AmazonBedrockAgentCoreSDKRuntime-us-east-1-8556fc4504
Creating IAM role: AmazonBedrockAgentCoreSDKRuntime-us-east-1-8556fc4504
✓ Role created: arn:aws:iam::710560201993:role/AmazonBedrockAgentCoreSDKRuntime-us-east-1-8556fc4504
✓ Execution policy attached: BedrockAgentCoreRuntimeExecutionPolicy-strands_claude_getting_started
Role creation complete and ready for use with Bedrock AgentCore
✅ Execution role available: arn:aws:iam::710560201993:role/AmazonBedrockAgentCoreSDKRunt

### Checking for the AgentCore Runtime Status
### 检查 AgentCore Runtime 状态

Now that we've deployed the AgentCore Runtime, let's check for it's deployment status

现在我们已经部署了 AgentCore Runtime，让我们检查一下它的部署状态。

In [15]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

Retrieved Bedrock AgentCore status for: strands_claude_getting_started


'READY'

### Invoking AgentCore Runtime
### 调用 AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

最后，我们可以使用有效负载调用 AgentCore Runtime。

<div style="text-align:left">
    <img src="images/invoke.png" width=75%"/>
</div>

In [16]:
invoke_response = agentcore_runtime.invoke({"prompt": "How is the weather now?"})
invoke_response

{'ResponseMetadata': {'RequestId': 'b33b286b-b776-4e0f-91ed-0985b496b02c',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Mon, 19 Jan 2026 09:49:53 GMT',
   'content-type': 'application/json',
   'transfer-encoding': 'chunked',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'b33b286b-b776-4e0f-91ed-0985b496b02c',
   'baggage': 'Self=1-696dfe3f-12d100b86727bfcb7a679229,session.id=044eed01-a94d-465e-960f-2d94ecaf7eb2',
   'x-amzn-bedrock-agentcore-runtime-session-id': '044eed01-a94d-465e-960f-2d94ecaf7eb2',
   'x-amzn-trace-id': 'Root=1-696dfe3f-03ec828a0bf2247f3dc7d441;Parent=4bea65d938b2051e;Sampled=1;Self=1-696dfe3f-12d100b86727bfcb7a679229'},
  'RetryAttempts': 0},
 'runtimeSessionId': '044eed01-a94d-465e-960f-2d94ecaf7eb2',
 'traceId': 'Root=1-696dfe3f-03ec828a0bf2247f3dc7d441;Parent=4bea65d938b2051e;Sampled=1;Self=1-696dfe3f-12d100b86727bfcb7a679229',
 'baggage': 'Self=1-696dfe3f-12d100b86727bfcb7a679229,session.id=044eed01-a94d-465e-960f-2d94ecaf7eb2',
 'contentType': 

### Processing invocation results
### 处理调用结果

We can now process our invocation results to include it in an application

现在我们可以处理调用结果，将其包含到应用程序中。

In [17]:
from IPython.display import Markdown, display
import json
response_text = invoke_response['response'][0]
display(Markdown(response_text))

The weather is sunny now! 🌞

### Invoking AgentCore Runtime with boto3
### 使用 boto3 调用 AgentCore Runtime

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

现在您的 AgentCore Runtime 已创建，您可以使用任何 AWS SDK 来调用它。例如，您可以使用 boto3 的 `invoke_agent_runtime` 方法。

In [18]:
import boto3
agent_arn = launch_result.agent_arn
agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 2+2?"})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

The result of 2 + 2 is 4.

## Cleanup (Optional)
## 清理（可选）

Let's now clean up the AgentCore Runtime created

现在让我们清理已创建的 AgentCore Runtime。

In [19]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split('/')[1]

('710560201993.dkr.ecr.us-east-1.amazonaws.com/bedrock-agentcore-strands_claude_getting_started',
 'strands_claude_getting_started-QEcvSFHhBL',
 'bedrock-agentcore-strands_claude_getting_started')

In [25]:
region

'us-east-1'

In [ ]:
agentcore_control_client = boto3.client(
    'bedrock-agentcore-control',
    region_name=region
)
ecr_client = boto3.client(
    'ecr',
    region_name=region
    
)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
    
)

response = ecr_client.delete_repository(
    repositoryName=launch_result.ecr_uri.split('/')[1],
    force=True
)

# Congratulations!
# 恭喜！